# 98. Validate Binary Search Tree
**Difficulty:** 🟡 Medium · **Topic:** Tree · **LeetCode:** https://leetcode.com/problems/validate-binary-search-tree/

## 💡 Concepts

**Core concept(s):** Check the **BST order rule** with bounds during DFS; or do an **in-order** walk and confirm it's sorted.

**Why it applies here:** A BST isn't just "each node bigger than its left child" — every node must fit within a range set by all its ancestors. Passing that range down (or checking the in-order sequence is strictly increasing) validates it.

**Key intuition:** Each node must fall inside a (low, high) window that tightens as you go down; or the in-order values must strictly increase.

---

### 📚 What is a Binary Tree?
A **binary tree** is nodes in a branching shape: each node holds a value and up to two children (**left**, **right**). The top is the **root**; childless nodes are **leaves**; **height** is the longest root-to-leaf path.
- **In Python:** a small `TreeNode` class with `.val`, `.left`, `.right`.

### 📚 What is a Binary Search Tree (BST)?
A **BST** stays ordered: for every node, everything in its **left** subtree is smaller and everything in its **right** subtree is larger — so you can find values by going left/right like binary search.
- **Key fact:** an **in-order** walk of a BST visits the values in **sorted** order.

### 📚 What is DFS (Depth-First Search) / Recursion?
**DFS** dives down one branch as far as possible, then backtracks. It's usually written with **recursion** — a function that calls itself on each child.
- **Complexity:** visits each node once → **O(n)** time; uses call-stack space up to the tree's **height**.

---

**Prerequisite knowledge:**
- Recursion carrying bounds.
- In-order traversal.

## 📝 Problem

Return `True` if the tree is a valid BST (left subtree all smaller, right subtree all larger, everywhere).

**Example**
```
[2,1,3]         -> True
[5,1,4,None,None,3,6] -> False   (4 is in 5's left subtree)
```

> Two approaches, both `O(n)`: range-bounds recursion and in-order check.

In [ ]:
from typing import Optional, List
from collections import deque

class TreeNode:
    """A single node of a binary tree: a value plus links to up to two children."""
    def __init__(self, val=0, left=None, right=None):
        self.val = val                     # the number stored at this node
        self.left = left                   # the left child (or None)
        self.right = right                 # the right child (or None)

def build_tree(values):
    """Build a tree from a level-order list, LeetCode style (None = missing child)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); q = deque([root]); i = 1
    while q and i < len(values):
        node = q.popleft()                 # the parent we're attaching children to
        if i < len(values):                # attach the left child (if present)
            if values[i] is not None:
                node.left = TreeNode(values[i]); q.append(node.left)
            i += 1
        if i < len(values):                # attach the right child (if present)
            if values[i] is not None:
                node.right = TreeNode(values[i]); q.append(node.right)
            i += 1
    return root

def build_balanced(n):
    """Balanced BST holding 1..n (height ~log n) — used by the benchmark."""
    def helper(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2               # middle value becomes the subtree's root
        node = TreeNode(mid)
        node.left = helper(lo, mid - 1)    # smaller values go left
        node.right = helper(mid + 1, hi)   # larger values go right
        return node
    return helper(1, n)

def preorder(root):
    """Collect values in preorder: node, then left, then right."""
    out = []
    def go(n):
        if not n: return
        out.append(n.val); go(n.left); go(n.right)
    go(root); return out

def inorder(root):
    """Collect values in inorder: left, then node, then right (sorted for a BST)."""
    out = []
    def go(n):
        if not n: return
        go(n.left); out.append(n.val); go(n.right)
    go(root); return out

def same_shape(a, b):
    """True if two trees have identical shape and values."""
    if not a and not b: return True        # both empty -> match
    if not a or not b or a.val != b.val: return False  # one empty, or values differ
    return same_shape(a.left, b.left) and same_shape(a.right, b.right)

### Approach 1 — Range Bounds (recursion)

**Idea:** Each node must be strictly inside `(low, high)`. Going left tightens the high bound to the node's value; going right tightens the low bound.

**Time:** `O(n)`. **Space:** `O(h)`.

In [ ]:
def is_valid_bounds(root: Optional[TreeNode]) -> bool:
    def valid(node, low, high):            # node's value must stay strictly inside (low, high)
        if not node:
            return True                    # empty subtree is fine
        if not (low < node.val < high):    # out of its allowed range -> not a BST
            return False
        # Going left tightens the upper bound; going right tightens the lower bound.
        return valid(node.left, low, node.val) and valid(node.right, node.val, high)
    return valid(root, float("-inf"), float("inf"))

### Approach 2 — In-Order Must Be Sorted

**Idea:** An in-order walk of a BST yields sorted values. Walk in-order (iteratively) and fail if any value isn't strictly greater than the previous.

**Time:** `O(n)`. **Space:** `O(h)`.

In [ ]:
def is_valid_inorder(root: Optional[TreeNode]) -> bool:
    stack, prev, node = [], float("-inf"), root   # prev = last value visited in order
    while stack or node:
        while node:                        # go as far left as possible first
            stack.append(node); 
            node = node.left
        node = stack.pop()                 # visit the smallest unvisited node
        if node.val <= prev:               # values must strictly increase in a BST
            return False
        prev = node.val                    # remember this value for the next comparison
        node = node.right                  # then explore the right subtree
    return True

In [ ]:
# Correctness check
tests = [([2,1,3],True), ([5,1,4,None,None,3,6],False), ([1],True), ([5,4,6,None,None,3,7],False)]
for vals, exp in tests:
    root = build_tree(vals)
    a, b = is_valid_bounds(root), is_valid_inorder(root)
    print(f"{vals} -> bounds={a}, inorder={b} | expected={exp}")
    assert a == b == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on trees of growing size `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(log n)`    | ≈ **1×** |
| `O(n)`        | ≈ **2×** |
| `O(n²)`       | ≈ **4×** |

We use **balanced** trees (height ~log n) so deep recursion stays safe while every node is still visited.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    return (build_balanced(n),)   # a valid BST -> full traversal
solutions = {
    "bounds  O(n)": is_valid_bounds,
    "inorder O(n)": is_valid_inorder,
}
sizes = [1000, 2000, 4000, 8000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Pass constraints down the recursion:** a node's validity depends on all its ancestors, not just its parent — carry a range.
- **In-order = sorted for a BST:** a powerful shortcut for many BST problems.
- **Signal:** "valid BST", "is this ordered correctly".
- **Related problems:** Kth Smallest in BST, Recover BST, Range Sum of BST.
- **Common pitfalls:** (1) only comparing a node to its direct children; (2) using `<=` vs `<` wrong (duplicates aren't allowed).